# Adi2 - From raw sources to a business answer

**Plan (120 min):** setup/API 10 | extract+profile 20 | clean+rules 30 | final table+recon 20 | answer 20 | deliverable+NOTES 20

**Assumptions log** (copy to NOTES.md with impact in rows / EUR / %):
- 

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.options.display.float_format = "{:,.2f}".format

API_URL = "http://localhost:8000"
RAW, CLEAN = Path("data/raw"), Path("data/clean")
RAW.mkdir(parents=True, exist_ok=True); CLEAN.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    "retail_sales": "retail/sales",
    "ecom_sales":   "ecom/sales",
    "stores":       "stores",
    "articles":     "articles",
    "fx_rates":     "fx_rates",
}
print(requests.get(f"{API_URL}/health").json())

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000191D7932590>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))

## 1. Extract (complete pagination, raw saved untouched)

In [ ]:
def fetch_all(endpoint, params=None, page_size=2000):
    params, rows, skip, total = dict(params or {}), [], 0, None
    while total is None or skip < total:
        r = requests.get(f"{API_URL}/{endpoint}",
                         params={**params, "limit": page_size, "skip": skip}, timeout=60)
        r.raise_for_status()
        js = r.json()
        total = js["total"]
        rows.extend(js["data"])
        if js["count"] == 0:
            break
        skip += js["count"]
    assert len(rows) == total, f"{endpoint}: got {len(rows)} of {total}"
    return pd.DataFrame(rows)

raw = {}
for name, ep in ENDPOINTS.items():
    raw[name] = fetch_all(ep)
    raw[name].to_parquet(RAW / f"{name}.parquet", index=False)
    print(f"{name:14s} {len(raw[name]):>8,} rows")

## 2. Profile (what does one row represent?)

In [ ]:
def profile(df, name="", cats=(), keys=None):
    print(f"=== {name} === shape={df.shape}")
    print(pd.concat([df.dtypes.rename("dtype"), df.isna().sum().rename("nulls"),
                     df.nunique().rename("unique")], axis=1).to_string())
    print("full-row duplicates:", df.duplicated().sum())
    if keys:
        print("duplicates on business key", keys, ":", df.duplicated(keys).sum())
    for c in cats:
        print(f"\n{c}:\n{df[c].value_counts(dropna=False).head(12).to_string()}")
    return df.head(3)

profile(raw["retail_sales"], "retail", cats=["transaction_type", "reversal_indicator", "country", "currency"],
        keys=["transaction_number", "line_number"])

In [ ]:
profile(raw["ecom_sales"], "ecom", cats=["transaction_type", "reversal_indicator", "fulfillment_channel",
                                         "order_origin", "country", "currency"],
        keys=["ecom_reference_number", "sor_itm"])

In [ ]:
for n in ["stores", "articles", "fx_rates"]:
    display(profile(raw[n], n))

In [ ]:
# TODO exploration: how do reversals (X) relate to original_tran_number? is the original still present?
#                   do RETURN rows carry negative or positive amounts? is net_sales = sold - returned?
#                   STORE orders without store_spot? DC orders with store_spot? stores missing in master?
#                   calendar_date vs consumer_order_date: how many rows change month?

## 3. Clean each source (one function per source, returns a DataFrame, no globals)

In [ ]:
def clean_stores(df):
    df = df.copy()
    df["store_spot"] = df["store_spot"].astype(str).str.strip().str.zfill(6)
    # TODO: duplicates in master? country consistent? names trimmed?
    return df.drop_duplicates("store_spot")

def clean_articles(df):
    df = df.copy()
    df["group_article"] = df["group_article"].astype(str).str.strip().str.upper()
    # TODO: duplicates? null division/category?
    return df.drop_duplicates("group_article")

def clean_fx(df):
    df = df.copy()
    assert df["units_per_eur"].gt(0).all()
    return df  # EUR = local / units_per_eur

stores, articles, fx = clean_stores(raw["stores"]), clean_articles(raw["articles"]), clean_fx(raw["fx_rates"])
fx

In [ ]:
COMMON_COLS = ["source", "order_id", "line_id", "transaction_type", "reversal_indicator", "original_tran_number",
               "date", "country", "store_spot", "group_article", "size_page",
               "sold_qty", "return_qty", "return_value", "net_qty", "net_sales", "currency",
               "fulfillment_channel", "order_origin"]

def _common_cleaning(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["calendar_date"], errors="coerce")   # TODO: mixed formats? nulls?
    df["group_article"] = df["group_article"].astype(str).str.strip().str.upper()
    df["country"] = df["country"].astype(str).str.strip().str.upper()
    for c in ["sold_qty", "return_qty", "return_value", "net_qty", "net_sales"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

def clean_retail(df):
    df = _common_cleaning(df)
    df["source"] = "retail"
    df["order_id"], df["line_id"] = df["transaction_number"].astype(str), df["line_number"]
    df["store_spot"] = df["store_spot"].astype(str).str.strip().str.zfill(6)
    df["fulfillment_channel"] = pd.NA; df["order_origin"] = pd.NA
    # TODO: duplicates on (transaction_number, line_number)?
    return df[COMMON_COLS]

def clean_ecom(df):
    df = _common_cleaning(df)
    df["source"] = "ecom"
    df["order_id"], df["line_id"] = df["ecom_reference_number"].astype(str), df["sor_itm"]
    df["store_spot"] = df["store_spot"].astype("string").str.strip().str.zfill(6)
    df.loc[df["store_spot"].isin(["<NA>", "000NAN", "0000NA"]), "store_spot"] = pd.NA  # TODO: check real null patterns
    # TODO: duplicates on (ecom_reference_number, sor_itm)?
    # TODO: decide date: calendar_date (same as retail) vs consumer_order_date; quantify rows that change month
    return df[COMMON_COLS]

retail, ecom = clean_retail(raw["retail_sales"]), clean_ecom(raw["ecom_sales"])
for n, d in {"stores": stores, "articles": articles, "fx": fx, "retail": retail, "ecom": ecom}.items():
    d.to_parquet(CLEAN / f"{n}.parquet", index=False)

## 4. Business rules (void, channel, FX), with a reconciliation log at every step

In [ ]:
recon = []
def log_step(step, df, to_eur=True):
    d = df.copy()
    d = d.merge(fx[["country", "units_per_eur"]], on="country", how="left")
    d["net_eur"] = d["net_sales"] / d["units_per_eur"]
    g = d.groupby("source").agg(rows=("net_sales", "size"), net_qty=("net_qty", "sum"), net_eur=("net_eur", "sum"))
    g.insert(0, "step", step)
    recon.append(g.reset_index())

sales = pd.concat([retail, ecom], ignore_index=True)
log_step("0. clean sources", sales)

In [ ]:
def drop_voids(df):
    # TODO: identify BOTH the reversal lines (X) and the original transactions they cancel
    #       (join on original_tran_number). Count and quantify each group separately before dropping.
    #       Careful: RETURN lines also use original_tran_number but are NOT voids.
    return df

sales = drop_voids(sales)
log_step("1. excluded voids (reversal + original)", sales)

In [ ]:
def assign_channel(df):
    df = df.copy()
    is_store_order = (df["source"] == "ecom") & (df["fulfillment_channel"] == "STORE")
    df["fulfilled_by_store_flag"] = is_store_order
    df["channel"] = np.where(df["source"] == "retail", "RETAIL", np.where(is_store_order, "RETAIL", "ECOM"))
    # TODO: edge cases: STORE without store_spot, DC with store_spot, store_spot not in master
    return df

sales = assign_channel(sales)
log_step("2. channel assignment (ship-from-store -> RETAIL)", sales)
pd.crosstab([sales["source"], sales["fulfillment_channel"].fillna("n/a")], sales["channel"])

## 5. Final table `fact_sellout_performance`

In [ ]:
def checked_merge(left, right, on, how="left", name=""):
    n0, s0 = len(left), left["net_sales"].sum()
    out = left.merge(right, on=on, how=how, validate="m:1")
    assert len(out) == n0, f"{name}: rows {n0} -> {len(out)}"
    assert np.isclose(out["net_sales"].sum(), s0), f"{name}: sales changed"
    print(f"{name}: rows {n0:,} -> {len(out):,} OK | unmatched keys: {out[on].isna().sum() if False else 'see below'}")
    return out

def build_fact(df, stores, articles, fx):
    d = checked_merge(df, fx[["country", "units_per_eur"]], on="country", name="fx")
    d = checked_merge(d, articles[["group_article", "product_division", "key_category"]], on="group_article", name="articles")
    d = d.merge(stores[["store_spot", "store_name"]], on="store_spot", how="left", validate="m:1")
    # TODO: orphans (store/article without master): keep as 'UNKNOWN' and report counts, do not drop silently
    # TODO: store_spot only for RETAIL rows (channel RETAIL) and ship-from-store; empty for DC
    d["month"] = d["date"].dt.strftime("%Y-%m")
    # TODO: gross vs return: verify gross_local = net_sales + return_value (or sold_qty based) on real data
    d["gross_sales_local"] = d["net_sales"] + d["return_value"]
    d["return_sales_local"] = d["return_value"]
    d["gross_qty"] = d["sold_qty"]
    for c in ["gross_sales", "return_sales"]:
        d[f"{c}_eur"] = d[f"{c}_local"] / d["units_per_eur"]
    d["net_sales_eur"] = d["gross_sales_eur"] - d["return_sales_eur"]
    grain = ["month", "country", "channel", "store_spot", "store_name", "group_article",
             "product_division", "key_category", "fulfilled_by_store_flag"]
    out = (d.groupby(grain, dropna=False, as_index=False)
             .agg(gross_qty=("gross_qty", "sum"), return_qty=("return_qty", "sum"),
                  gross_sales_eur=("gross_sales_eur", "sum"), return_sales_eur=("return_sales_eur", "sum"),
                  net_sales_eur=("net_sales_eur", "sum")))
    out["net_qty"] = out["gross_qty"] - out["return_qty"]
    return out

fact = build_fact(sales, stores, articles, fx)
fact.to_parquet("fact_sellout_performance.parquet", index=False)
fact.shape

## 6. Validation and reconciliation

In [ ]:
recon_df = pd.concat(recon, ignore_index=True)
log_final = (fact.assign(source=np.where(fact["fulfilled_by_store_flag"], "ecom", fact["channel"].str.lower()))
                 .groupby("source").agg(rows=("net_sales_eur", "size"), net_qty=("net_qty", "sum"),
                                        net_eur=("net_sales_eur", "sum")).reset_index())
log_final.insert(0, "step", "9. fact_sellout_performance (aggregated)")
display(pd.concat([recon_df, log_final]).sort_values(["source", "step"]))

grain = ["month", "country", "channel", "store_spot", "group_article", "fulfilled_by_store_flag"]
assert not fact.duplicated(grain).any(), "duplicate keys at grain"
assert fact["net_sales_eur"].notna().all()
# TODO: net = gross - returns check; every row difference in the table above explained in NOTES.md

## 7. Business question (only from `fact`)
**Q1 2026 vs Q1 2025, EUR.** (1) Is the data consistent with "growth is coming from online, stores are losing relevance"? (2) Which stores drive Retail and why?

In [ ]:
f = pd.read_parquet("fact_sellout_performance.parquet")
f["year"] = f["month"].str[:4]
q1 = f[f["month"].str[5:7].isin(["01", "02", "03"]) & f["year"].isin(["2025", "2026"])]

def growth(df, by):
    t = df.pivot_table(index=by, columns="year", values="net_sales_eur", aggfunc="sum", fill_value=0)
    t["delta_eur"] = t["2026"] - t["2025"]
    t["growth_%"] = np.where(t["2025"] != 0, t["delta_eur"] / t["2025"] * 100, np.nan)
    return t

display(growth(q1, ["channel"]))
display(growth(q1, ["country", "channel"]))
# Key comparison: "naive" view (ECOM source as-is) vs business-rule view (ship-from-store -> RETAIL)
display(growth(q1.assign(view=np.where(q1["fulfilled_by_store_flag"], "ECOM (ship-from-store)", q1["channel"])), ["view"]))

In [ ]:
# Q2: stores driving Retail. TODO: like-for-like (stores active in both periods) vs new stores,
#     ship-from-store share per store, store_format (needs stores master), concentration (top 10 share)
retail_q1 = q1[q1["channel"] == "RETAIL"]
stores_g = growth(retail_q1, ["store_spot", "store_name"]).sort_values("delta_eur")
display(stores_g.head(10)); display(stores_g.tail(10))

## 8. Charts for leadership

In [ ]:
ch = growth(q1, ["country", "channel"])[["2025", "2026"]].reset_index()
fig, ax = plt.subplots(figsize=(9, 4))
# TODO: clear title = the message (not the topic), labels in EUR M, one chart per message
for i, (k, g) in enumerate(ch.groupby("channel")):
    ax.bar(np.arange(len(g)) + i * 0.35, g["2026"] / g["2025"] - 1, width=0.35, label=k)
ax.set_xticks(np.arange(len(g)) + 0.17); ax.set_xticklabels(g["country"])
ax.axhline(0, color="k", lw=.5); ax.legend(); ax.set_ylabel("Net sales growth Q1'26 vs Q1'25")
plt.tight_layout(); plt.savefig("chart_growth.png", dpi=150)

## 9. Insights (what I observe -> what it means -> what I would do)
1. 
2. 
3. 

## 10. Before delivering
- Kernel > Restart & Run All works top to bottom
- `NOTES.md` filled (how to run, decisions with impact, data issues, AI usage)
- Zip: notebook, `data/clean/`, `fact_sellout_performance.parquet`, charts / deck, `NOTES.md`